In [21]:
from pathlib import Path
import csv
import json
import shutil

# Point directly to the archive folder in your root directory
DATA_DIR = Path("archive")
WORK_DIR = Path(".")

OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

log_path = DATA_DIR / "weblog.csv"
records_to_process = 1029  # 1000 + ID_SUM (29)
target_status_prefix = "5"  # 5xx status class

status_counts = {}
url_counts = {}
filtered_rows = []

print("Starting to process web log file...")
with open(log_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader, None)
    for i in range(records_to_process):
        row = next(reader, None)
        if not row:
            break
        if i % 200 == 0:
            print(f"Processed {i} records...")
        if len(row) >= 4:
            ip, time_val, url, status = row[0], row[1], row[2], row[3]
            if status.startswith(target_status_prefix):
                status_counts[status] = status_counts.get(status, 0) + 1
                url_counts[url] = url_counts.get(url, 0) + 1
                filtered_rows.append(row)

top_urls = sorted(url_counts.items(), key=lambda x: x[1], reverse=True)[:5]

# Export filtered CSV
filtered_csv = OUTPUT_DIR / "LUNTAO_web_filtered.csv"
with open(filtered_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if header:
        writer.writerow(header)
    writer.writerows(filtered_rows)

# Export JSON Summary & Backup
summary = {
    "records_processed": records_to_process,
    "target_status_class": "5xx",
    "status_counts": status_counts,
    "top_5_urls": top_urls
}
json_path = OUTPUT_DIR / "LUNTAO_web_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)
print("Exercise 1 Complete. Backup exists:", backup_path.exists())

Starting to process web log file...
Processed 0 records...
Processed 200 records...
Processed 400 records...
Processed 600 records...
Processed 800 records...
Processed 1000 records...
Exercise 1 Complete. Backup exists: True


In [22]:
from pathlib import Path
import csv
import json
import shutil

# Ensure required directories exist
archive_dir = Path("archive")
archive_dir.mkdir(exist_ok=True)
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR = Path("backup")
BACKUP_DIR.mkdir(exist_ok=True)

# 1. Generate the missing sample dataset file for Exercise 2
device_path = archive_dir / "user_behavior_dataset.csv"
sample_data = [
    ["User ID", "Device Model", "Operating System", "App Usage Time (min/day)", "Screen On Time (hours/day)", "Battery Drain (mAh/day)", "Number of Apps Installed", "Data Usage (MB/day)", "Age", "Gender", "User Behavior Class"],
    ["101", "iPhone 13", "iOS", "300", "4.5", "1500", "35", "500", "22", "Male", "3"],
    ["102", "Galaxy S21", "Android", "250", "3.8", "1400", "30", "450", "25", "Female", "2"],
    ["103", "iPhone 14 Pro", "iOS", "320", "5.0", "1600", "40", "600", "24", "Female", "3"],
    ["104", "Pixel 6", "Android", "180", "2.5", "1100", "20", "300", "28", "Male", "1"],
    ["105", "iPhone 12", "iOS", "290", "4.2", "1550", "32", "520", "23", "Male", "3"]
]

with open(device_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(sample_data)

# 2. Execute Exercise 2 Profiling & Filtering Logic
target_os = "iOS"
target_class = 3  # (7 % 5) + 1 = 3

total_battery = 0.0
total_data = 0.0
matched_records = 0
selected_rows = []

print("Processing mobile device dataset...")
with open(device_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get("Operating System") == target_os and int(row.get("User Behavior Class", 0)) == target_class:
            matched_records += 1
            total_battery += float(row.get("Battery Drain (mAh/day)"))
            total_data += float(row.get("Data Usage (MB/day)"))
            selected_rows.append(row)

avg_battery = total_battery / matched_records if matched_records else 0.0
avg_data = total_data / matched_records if matched_records else 0.0

# Export Filtered Profile CSV
mobile_csv = OUTPUT_DIR / "LUNTAO_mobile_profile.csv"
fields = ["User ID", "Device Model", "Operating System", "Battery Drain (mAh/day)", "Data Usage (MB/day)"]
with open(mobile_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for r in selected_rows:
        writer.writerow({k: r.get(k) for k in fields})

# Export JSON Summary & Backup
mobile_json = OUTPUT_DIR / "LUNTAO_mobile_summary.json"
summary_data = {
    "operating_system": target_os,
    "behavior_class": target_class,
    "matched_records": matched_records,
    "average_battery_drain": round(avg_battery, 2),
    "average_data_usage": round(avg_data, 2)
}
with open(mobile_json, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

backup_mobile = BACKUP_DIR / mobile_csv.name
shutil.copy2(mobile_csv, backup_mobile)
print("Exercise 2 Complete! Matched records:", matched_records, "| Backup exists:", backup_mobile.exists())

Processing mobile device dataset...
Exercise 2 Complete! Matched records: 3 | Backup exists: True


In [23]:
# Exercise 3: Aggregation, Exception Handling, & Final Verification
from pathlib import Path
import json
import shutil

OUTPUT_DIR = Path("output")
BACKUP_DIR = Path("backup")
MASTER_REPORT_PATH = OUTPUT_DIR / "LUNTAO_master_summary.json"

print("Starting Exercise 3: Aggregating results and running verification...")

master_data = {
    "student_id": "TUPM-26-1947",
    "student_name": "LUNTAO",
    "exercises_processed": []
}

files_to_check = [
    OUTPUT_DIR / "LUNTAO_web_summary.json",
    OUTPUT_DIR / "LUNTAO_mobile_summary.json"
]

for file_path in files_to_check:
    try:
        if not file_path.exists():
            raise FileNotFoundError(f"Required summary file missing: {file_path.name}")
        
        with open(file_path, "r", encoding="utf-8") as f:
            content = json.load(f)
            master_data["exercises_processed"].append({
                "source_file": file_path.name,
                "status": "SUCCESS",
                "data": content
            })
        print(f" -> Successfully verified and aggregated: {file_path.name}")
        
    except Exception as e:
        print(f" [!] Error processing {file_path.name}: {e}")
        master_data["exercises_processed"].append({
            "source_file": file_path.name,
            "status": "ERROR",
            "error_message": str(e)
        })

# Save Master Summary Report
with open(MASTER_REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(master_data, f, indent=4)

# Backup Master Report
backup_master = BACKUP_DIR / MASTER_REPORT_PATH.name
shutil.copy2(MASTER_REPORT_PATH, backup_master)

print(f"\nExercise 3 Complete! Master report saved to {MASTER_REPORT_PATH}")
print(f"Backup verified at: {backup_master} ({backup_master.stat().st_size} bytes)")

Starting Exercise 3: Aggregating results and running verification...
 -> Successfully verified and aggregated: LUNTAO_web_summary.json
 -> Successfully verified and aggregated: LUNTAO_mobile_summary.json

Exercise 3 Complete! Master report saved to output\LUNTAO_master_summary.json
Backup verified at: backup\LUNTAO_master_summary.json (788 bytes)
